# **Урок 4**. Разбираемся в ViT 


Из лекции мы с вами узнали, что в ```ViT``` авторы перенесли стандартный ```Transformer Encoder``` из ```NLP``` в ```CV``` и получили довольно классные результаты. Давайте мы сегодня попробуем разобраться с базовым кодом ```ViT```-а и его работой с картинками. 

## **План**

1. Собираем ```Transformer```
2. Собираем **Vision** ```Transformer```

Погнали!

## **1. Собираем Transformer**

<img src="images/transformer.PNG" width="200" />

Итак, ```Transformer block``` состоит из:
- ```Layer Normalization```
- ```Multi-Head Self-Attention```
- ```Feed Forward/MLP```
- ```Skip Connections```

Давайте собирать его из кусочков!

[Оригинальная реализация]((https://github.com/google-research/vision_transformer)) авторов была сделана на фреймворке jax от Google, что нам, как любителями pytorch, не очень подходит. 

В ```torchvision```, конечно, тоже [уже есть реализация](https://pytorch.org/vision/stable/models/vision_transformer.html), вы можете её использовать со стандартными командами, но её [source code](https://pytorch.org/vision/main/_modules/torchvision/models/vision_transformer.html) не очень friendly, а нам пока хотелось бы основные концепты понять.  

Поэтому разбираться мы будем с ```open-source``` имплементацией ```ViT``` на ```pytorch``` - [vit_pytorch](https://github.com/lucidrains/vit-pytorch/tree/main).

Здесь для реализации архитектуры используется [```einops```](https://github.com/arogozhnikov/einops/tree/master) - библиотека для эффективных операций над тензорами. 

Оттуда используются методы ```rearrange```-  аналог ```reshape``` (он же используется ещё и в виде ```nn``` слоя в самой архитектуре), и ```repeat``` - дублирующий тензор вдоль выбранной оси. 

In [1]:
import torch
from torch import nn

from einops import rearrange, repeat
from einops.layers.torch import Rearrange

Начнем с самого простого - ```MLP```. Это просто маленькая полносвязная сеточка. 

In [2]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim), # в начало MLP суем LayerNorm просто как слой
            
            nn.Linear(dim, hidden_dim),
            nn.GELU(), # это Gaussian ELU https://pytorch.org/docs/stable/generated/torch.nn.GELU.html
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, dim), # начальная и конечная размерность - dim
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

Вспоминаем, что там происходит в ```Attention```:

<img src="images/attention.png" width="500" />

Теперь посмотрим на ```Multi-Head Self-Attention```:

<img src="images/MHSA.png" width="1000" />

In [3]:
# dim - размерность каждого приходящего эмбеддинга
# heads - число голов в Attention
# dim_head - d_k, внутренняя размерность Attention, вторая размерность в матрицах W^Q, W^K, W^V и длина векторов q,k,v

class Attention(nn.Module):
    def __init__(self, dim, heads = 8, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = dim_head *  heads # это внутренняя размерность сразу для всех голов
        
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads # столько у нас параллельно Attention работает
        self.scale = dim_head ** -0.5 # это sqrt(d_k), на него будем делить q*k

        self.norm = nn.LayerNorm(dim)

        self.attend = nn.Softmax(dim = -1)
        self.dropout = nn.Dropout(dropout)

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False) # получим q,k,v для всех heads из x сразу за один линейный слой

        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

        
    def forward(self, x):
        print(f'Init x: {x.shape}')
        x = self.norm(x)  # Layer Norm в начале
        print(f'X after layer norm: {x.shape}')
        
        # получили за одну операцию q,k,v в виде матрицы 
        qkv = self.to_qkv(x)
        print(f'qkv: {qkv.shape}')
        
        # и разбили её на 3 части, https://pytorch.org/docs/stable/generated/torch.chunk.html
        qkv = qkv.chunk(3, dim=-1) 
        print(f'Shapes: q: {qkv[0].shape}, k: {qkv[1].shape}, v: {qkv[2].shape}')
        
        # разделили по отдельным heads значения
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)
        print(f'After rearrange: q: {q.shape}, k: {k.shape}, v: {v.shape}')
        
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale # умножили query на key, поделили на scale
        print(f'Query * Key / Scale: {dots.shape}')
        
        attn = self.attend(dots) # накинули softmax
        attn = self.dropout(attn)
        print(f'Attentions: {attn.shape}')
        
        out = torch.matmul(attn, v) # взвесили все value c весами attention
        print(f'Values with Attentions: {out.shape}')
        
        out = rearrange(out, 'b h n d -> b n (h d)')
        print(f'Final result: {out.shape}')
        
        return self.to_out(out)

In [5]:
att = Attention(512, heads = 8, dim_head = 64, dropout = 0.)
img = torch.randn(1, 15, 512)

res = att(img)

Init x: torch.Size([1, 15, 512])
X after layer norm: torch.Size([1, 15, 512])
qkv: torch.Size([1, 15, 1536])
Shapes: q: torch.Size([1, 15, 512]), k: torch.Size([1, 15, 512]), v: torch.Size([1, 15, 512])
After rearrange: q: torch.Size([1, 8, 15, 64]), k: torch.Size([1, 8, 15, 64]), v: torch.Size([1, 8, 15, 64])
Query * Key / Scale: torch.Size([1, 8, 15, 15])
Attentions: torch.Size([1, 8, 15, 15])
Values with Attentions: torch.Size([1, 8, 15, 64])
Final result: torch.Size([1, 15, 512])


Супер, соберем теперь эти два блока в Трансформер Энкодер - по сути, нам надо просто настакать друг за другом повторяющиеся блоки из ```Feed Forward + Attention```.

In [6]:
# dim - размерность эмбеддингов
# depth - число блоков трансформер энкодера
# heads - число голов в Attention
# dim_head - внутренняя размерность Attention-а
# mlp_dim - внутрення размерность MLP

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([])
        
        # набираем блоков столько, сколько в параметрах задали
        for _ in range(depth):
            # вот это и есть трансформер блоки
            self.layers.append(nn.ModuleList([
                Attention(dim, heads = heads, dim_head = dim_head, dropout = dropout),
                FeedForward(dim, mlp_dim, dropout = dropout)
            ]))

    def forward(self, x):
        # а тут у нас все блоки энкодера и их skip connections
        for attn, ff in self.layers:
            print('Encoder Block starts')
            x = attn(x) + x 
            x = ff(x) + x
            print('____________________')
        return x

Давайте срезюмируем:
- мы разобрались с ```Feed Forward```
- посмотрели, как в коде реализуется ```Multi-Head Self-Attention```
- собрали из этих кусочков (а ещё из ```Layer Norm``` и ```Skip Connections```) полноценный Transformer Encoder

## **2. Собираем Vision Transformer**

<img src="images/vit.png" width="800" />

Давайте вспомним основные шаги в работе ```ViT```:
1. Изображение нарезается на патчи
2. Патчи вытягиваются в вектора
3. Патчи-вектора превращаются в токены/эмбеддинги с помощью ```Linear Projection```
4. Добавляется ```<cls>``` эмбеддинг
5. Ко всем эмбеддингам добавляются позиционные эмбеддинги
6. Полученные эмбеддинги прогодят через несколько блоков типа ```Transformer Encoder``` (этот пункт у нас готов!)
7. Накопленный в ```<cls>``` эмбеддинг проходит через последнюю ```MLP Head/Feed Forward``` 
8. Получаем ```logit```-ы и выбираем нужный класс

Немножко страшно, но давайте собирать:

In [7]:
# эта функция вернет t, если t - уже tuple
# если нет, то составит tuple (t, t)
def pair(t):
    return t if isinstance(t, tuple) else (t, t)

In [8]:
# image_size - размер входного изображения, задаем через int или tuple, кстати img не обязательно квадратный
# patch_size - размер патча, задаем через int или tuple, кстати, патч не обязательно квадратный
# num_classes - число классов в задаче
# dim - размерность эмбеддингов для патчей
# depth - число энкодер блоков
# heads - число голов в MHSA
# mlp_dim - размерность эмбеддинга в MLP
# pool - как будем агрегировать знания из финальных эмбеддингов для классификации {'cls', 'mean'}
# channels - число каналов входящего изображения (обычно 3, но может быть и 1)
# dim_head - размерность q, k, v
# dropout, emb_dropout - вероятности dropout-ов
class ViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, pool='cls', channels=3, dim_head=64, dropout=0., emb_dropout=0.):
        super().__init__()
        # сразу проверяем что pool в нужном диапазоне
        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'
        
        # тут мы преобразуем входные параметры, чтобы они точно стали tuple - (t, t)
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)
        
        # провереям кратность размерностей
        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'
        
        # считаем число патчей
        num_patches = (image_height // patch_height) * (image_width // patch_width)
        
        # считаем flatten длину патча
        patch_dim = channels * patch_height * patch_width
        
        # Linear Projection - просто обучаемый FC
        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_height, p2 = patch_width), # тут мы сделали flatten патчей
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim), # преобразовали к рабочей размерности dim, она теперь с нами до конца
            nn.LayerNorm(dim),
        )
        
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim)) # генерируем num_patches + 1 случайных векторов длины dim из N(0, 1)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim)) # генерируем  случайный <cls> токен из N(0,1)
        self.dropout = nn.Dropout(emb_dropout)

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)

        self.pool = pool

        self.mlp_head = nn.Linear(dim, num_classes)

    def forward(self, img):
        x = self.to_patch_embedding(img) # Patching, Flatten, Linear Projection
        b, n, _ = x.shape # размерность батча, размерность числа патчей

        cls_tokens = repeat(self.cls_token, '1 1 d -> b 1 d', b = b) # сделали <cls> токен для каждой картинки в батче
        x = torch.cat((cls_tokens, x), dim=1) # приделываем всем <cls> токен
        x += self.pos_embedding[:, :(n + 1)] # добавляем всем позиционные эмбеддинги
        x = self.dropout(x)

        x = self.transformer(x) # гоняем через все трансформерные блоки
        
        # если выбрали mean, nо усредняем все эмбеддинги, а если нет, то берем только <cls>
        x = x.mean(dim = 1) if self.pool == 'mean' else x[:, 0]
        
        # финально классифицируем
        return self.mlp_head(x)

In [9]:
v = ViT(
    image_size = 256,
    patch_size = 32,
    num_classes = 1000,
    dim = 1024,
    depth = 4,
    heads = 8,
    mlp_dim = 2048,
    dropout = 0.1,
    emb_dropout = 0.1
)

img = torch.randn(1, 3, 256, 256)

preds = v(img)
print(preds.shape)

Encoder Block starts
Init x: torch.Size([1, 65, 1024])
X after layer norm: torch.Size([1, 65, 1024])
qkv: torch.Size([1, 65, 1536])
Shapes: q: torch.Size([1, 65, 512]), k: torch.Size([1, 65, 512]), v: torch.Size([1, 65, 512])
After rearrange: q: torch.Size([1, 8, 65, 64]), k: torch.Size([1, 8, 65, 64]), v: torch.Size([1, 8, 65, 64])
Query * Key / Scale: torch.Size([1, 8, 65, 65])
Attentions: torch.Size([1, 8, 65, 65])
Values with Attentions: torch.Size([1, 8, 65, 64])
Final result: torch.Size([1, 65, 512])
____________________
Encoder Block starts
Init x: torch.Size([1, 65, 1024])
X after layer norm: torch.Size([1, 65, 1024])
qkv: torch.Size([1, 65, 1536])
Shapes: q: torch.Size([1, 65, 512]), k: torch.Size([1, 65, 512]), v: torch.Size([1, 65, 512])
After rearrange: q: torch.Size([1, 8, 65, 64]), k: torch.Size([1, 8, 65, 64]), v: torch.Size([1, 8, 65, 64])
Query * Key / Scale: torch.Size([1, 8, 65, 65])
Attentions: torch.Size([1, 8, 65, 65])
Values with Attentions: torch.Size([1, 8, 65,

Супер, вот мы и разобрались во всём :)

Конечно, в реальных задачах советую брать реализованную и отлаженную архитектуру из крупных бибилиотек. 

Например, так:

In [10]:
import timm

# так можно посмотреть, что есть ViT-подобное
print(timm.list_models("*vit*"), len(timm.list_models("*vit*")))

# выбираем из списочка и создаем
vit_timm = timm.create_model("vit_base_patch16_224")

/ssd/a.belozerova/usr/anaconda3/envs/torch24/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['convit_base', 'convit_small', 'convit_tiny', 'crossvit_9_240', 'crossvit_9_dagger_240', 'crossvit_15_240', 'crossvit_15_dagger_240', 'crossvit_15_dagger_408', 'crossvit_18_240', 'crossvit_18_dagger_240', 'crossvit_18_dagger_408', 'crossvit_base_240', 'crossvit_small_240', 'crossvit_tiny_240', 'davit_base', 'davit_base_fl', 'davit_giant', 'davit_huge', 'davit_huge_fl', 'davit_large', 'davit_small', 'davit_tiny', 'efficientvit_b0', 'efficientvit_b1', 'efficientvit_b2', 'efficientvit_b3', 'efficientvit_l1', 'efficientvit_l2', 'efficientvit_l3', 'efficientvit_m0', 'efficientvit_m1', 'efficientvit_m2', 'efficientvit_m3', 'efficientvit_m4', 'efficientvit_m5', 'fastvit_ma36', 'fastvit_mci0', 'fastvit_mci1', 'fastvit_mci2', 'fastvit_s12', 'fastvit_sa12', 'fastvit_sa24', 'fastvit_sa36', 'fastvit_t8', 'fastvit_t12', 'flexivit_base', 'flexivit_large', 'flexivit_small', 'gcvit_base', 'gcvit_small', 'gcvit_tiny', 'gcvit_xtiny', 'gcvit_xxtiny', 'levit_128', 'levit_128s', 'levit_192', 'levit_256', 

Или [вот так](https://pytorch.org/vision/main/models/vision_transformer.html) по знакомому нам рецепту:

In [11]:
from torchvision import models

vit_torch = models.vit_b_16(weights='DEFAULT')

In [12]:
vit_torch

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

Вообще трансформеры довольно сложно обучать, и, как вы помните, им нужно много данных для того, чтобы работать хорошо. 

В рамках этого курса нам важно только понять и прочувствовать устройство этой модели, ведь у вас это могут спросить на собеседовании. 

Надеюсь, вы с этими вопросами легко справитесь!

## **Итоги**

Итак,  мы
1. Узнали, как реализуются в коде все основные части ```Transformer``` блока
2. Разобрали, как происходить патчизация, проекция и добавление позиционных эмбеддингов
3. Поняли, как все части собираются вместе и образуют ```Vision Transformer```


И это ещё не конец, ведь во второй практике мы разберемся, как работать с ```ViT```-ами, обученными с помощью ```CLIP```. 

See you!